In [11]:
import os
os.environ["ORT_DISABLE_THREAD_AFFINITY"] = "1"
import onnx
import onnxruntime as ort
ort.set_default_logger_severity(3) # Reduces noise (3 = Error)

import numpy as np
from pathlib import Path

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", ort.__version__)

ONNX version: 1.21.0
ONNX Runtime version: 1.26.0


In [2]:
file_path = Path("/global/cfs/cdirs/m3443/usr/xju/onnx_debug_models/GN3Large")
onnx_path = file_path / "GN3Large.onnx"

In [3]:
model = onnx.load(onnx_path)
onnx.checker.check_model(model)
print("The model is checked!")

The model is checked!


In [4]:
print(onnx.helper.printable_graph(model.graph))

graph main_graph (
  %jet_features[FLOAT, 1x2]
  %track_features[FLOAT, n_tracksx24]
  %flow_features[FLOAT, n_flowsx5]
  %electron_features[FLOAT, n_electronsx28]
) initializers (
  %model.init_nets.0.net.net.0.bias[FLOAT, 512]
  %model.init_nets.0.net.net.2.bias[FLOAT, 1024]
  %model.init_nets.1.net.net.0.bias[FLOAT, 512]
  %model.init_nets.1.net.net.2.bias[FLOAT, 1024]
  %model.init_nets.2.net.net.0.bias[FLOAT, 512]
  %model.init_nets.2.net.net.2.bias[FLOAT, 1024]
  %model.tasks.0.net.net.0.weight[FLOAT, 128x256]
  %model.tasks.0.net.net.0.bias[FLOAT, 128]
  %model.tasks.0.net.net.2.weight[FLOAT, 64x128]
  %model.tasks.0.net.net.2.bias[FLOAT, 64]
  %model.tasks.0.net.net.4.weight[FLOAT, 32x64]
  %model.tasks.0.net.net.4.bias[FLOAT, 32]
  %model.tasks.0.net.net.6.weight[FLOAT, 6x32]
  %model.tasks.0.net.net.6.bias[FLOAT, 6]
  %model.tasks.4.net.net.0.weight[FLOAT, 128x256]
  %model.tasks.4.net.net.0.bias[FLOAT, 128]
  %model.tasks.4.net.net.2.weight[FLOAT, 64x128]
  %model.tasks.4.ne

/tmp/ipykernel_265893/2222816960.py:1: DeprecationWarning: Deprecated since 1.19. Consider using onnx.printer.to_text() instead.
  print(onnx.helper.printable_graph(model.graph))


In [5]:
def inspect(model_path: str):
    model = onnx.load(model_path)
    onnx.checker.check_model(model)
    # onnx.helper.printable_graph(model.graph)

    sess_opts = ort.SessionOptions()
    sess_opts.enable_mem_reuse = False
    ort_sess = ort.InferenceSession(model_path, sess_opts=sess_opts)
    print("Input: ")
    for i in range(len(ort_sess.get_inputs())):
        print("\t", i, ": ", ort_sess.get_inputs()[i])
    
    print("Output: ")
    for i in range(len(ort_sess.get_outputs())):
        print("\t", i, ": ", ort_sess.get_outputs()[i])

    input_names = [x.name for x in ort_sess.get_inputs()]
    input_shapes = [x.shape for x in ort_sess.get_inputs()]
    output_names = [x.name for x in ort_sess.get_outputs()]
    output_shapes = [x.shape for x in ort_sess.get_outputs()]
    return input_names, input_shapes, output_names, output_shapes


In [6]:
ioshapes = inspect(onnx_path)

2026-05-26 10:05:34.020216197 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 266298, index: 2, mask: {3, 131, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-26 10:05:34.020216738 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 266296, index: 0, mask: {1, 129, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-26 10:05:34.020381134 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 266299, index: 3, mask: {4, 132, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-26 10:05:34.020220585 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 266297, index: 1, mask: {2, 130, }, error code: 22 error msg: Invalid argumen

Input: 
	 0 :  NodeArg(name='jet_features', type='tensor(float)', shape=[1, 2])
	 1 :  NodeArg(name='track_features', type='tensor(float)', shape=['n_tracks', 24])
	 2 :  NodeArg(name='flow_features', type='tensor(float)', shape=['n_flows', 5])
	 3 :  NodeArg(name='electron_features', type='tensor(float)', shape=['n_electrons', 28])
Output: 
	 0 :  NodeArg(name='GN3Large_pb', type='tensor(float)', shape=[])
	 1 :  NodeArg(name='GN3Large_pc', type='tensor(float)', shape=[])
	 2 :  NodeArg(name='GN3Large_pudjets', type='tensor(float)', shape=[])
	 3 :  NodeArg(name='GN3Large_pgjets', type='tensor(float)', shape=[])
	 4 :  NodeArg(name='GN3Large_psjets', type='tensor(float)', shape=[])
	 5 :  NodeArg(name='GN3Large_ptau', type='tensor(float)', shape=[])
	 6 :  NodeArg(name='GN3Large_ptFromTruthDressedWZJet', type='tensor(float)', shape=[])


In [7]:
n_tracks = 10
n_flows = 4
n_electrons = 1
jet_features = np.random.rand(1, 2).astype(np.float32)
track_features = np.random.rand(n_tracks, 24).astype(np.float32)
flow_features = np.random.rand(n_flows, 5).astype(np.float32)
electron_features = np.random.rand(n_electrons, 28).astype(np.float32)

In [16]:
input_names, input_shapes, output_names, output_shapes = ioshapes

options = ort.SessionOptions()
options.intra_op_num_threads = 1 
options.inter_op_num_threads = 1
# options.add_session_config_entry('session.intra_op_thread_affinities', '1;2')


providers = [
    ('CUDAExecutionProvider', {
        'device_id': 0,
        'arena_extend_strategy': 'kNextPowerOfTwo',
        # 'gpu_mem_limit': 2 * 1024 * 1024 * 1024,
        'cudnn_conv_algo_search': 'EXHAUSTIVE',
        'do_copy_in_default_stream': True,
    }),
    'CPUExecutionProvider',
]

ort_sess = ort.InferenceSession(onnx_path, sess_opts=options, providers=["CUDAExecutionProvider"])

results = ort_sess.run(output_names, 
                       {input_names[0]: jet_features, 
                        input_names[1]: track_features, 
                        input_names[2]: flow_features, 
                        input_names[3]: electron_features}
                    )

2026-05-26 10:10:39.133043334 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 281335, index: 0, mask: {1, 129, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-26 10:10:39.133048824 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 281336, index: 1, mask: {2, 130, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-26 10:10:39.133123273 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 281337, index: 2, mask: {3, 131, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-26 10:10:39.133148821 [E:onnxruntime:Default, env.cc:228 ThreadMain] pthread_setaffinity_np failed for thread: 281338, index: 3, mask: {4, 132, }, error code: 22 error msg: Invalid argumen

In [17]:
results

[array(0.9998227, dtype=float32),
 array(0.00014868, dtype=float32),
 array(1.2410574e-05, dtype=float32),
 array(1.6331962e-05, dtype=float32),
 array(3.7057855e-25, dtype=float32),
 array(1.5245638e-24, dtype=float32),
 array(1.269604, dtype=float32)]